# 06A - Logistic Regression (Baseline Model)

Enterprise baseline classification model.

In [1]:

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,
roc_auc_score,classification_report,confusion_matrix)

df=pd.read_csv("../data/datasets/american_bankruptcy.csv")
df['target']=df['status_label'].map({'alive':0,'failed':1})
drop=['status_label','target']
if 'company_name' in df.columns: drop.append('company_name')
X=df.drop(columns=drop)
y=df['target']
num=X.select_dtypes(include='number').columns
cat=X.select_dtypes(exclude='number').columns

pre=ColumnTransformer([
('num',Pipeline([('imp',SimpleImputer(strategy='median')),
                 ('sc',StandardScaler())]),num),
('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent'))]),cat)
])

X_train,X_test,y_train,y_test=train_test_split(
X,y,test_size=0.2,stratify=y,random_state=42)


## Train Baseline Model

In [2]:

model=Pipeline([
('preprocessor',pre),
('classifier',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42))
])
model.fit(X_train,y_train)
pred=model.predict(X_test)
proba=model.predict_proba(X_test)[:,1]


## Cross Validation

In [3]:

cv=cross_val_score(model,X_train,y_train,cv=5,scoring='f1')
print("F1 Scores:",cv)
print("Mean F1:",cv.mean().round(4))


F1 Scores: [0.17851337 0.16965889 0.17378584 0.17623942 0.17208129]
Mean F1: 0.1741


## Evaluation Metrics

In [4]:

metrics=pd.DataFrame({
'Metric':['Accuracy','Precision','Recall','F1','ROC-AUC'],
'Value':[
accuracy_score(y_test,pred),
precision_score(y_test,pred),
recall_score(y_test,pred),
f1_score(y_test,pred),
roc_auc_score(y_test,proba)
]
})
display(metrics)
print(classification_report(y_test,pred))
print("Confusion Matrix")
print(confusion_matrix(y_test,pred))


,Metric,Value
0,Accuracy,0.562115
1,Precision,0.099026
2,Recall,0.691571
3,F1,0.173245
4,ROC-AUC,0.659467


              precision    recall  f1-score   support

           0       0.96      0.55      0.70     14693
           1       0.10      0.69      0.17      1044

    accuracy                           0.56     15737
   macro avg       0.53      0.62      0.44     15737
weighted avg       0.90      0.56      0.67     15737

Confusion Matrix
[[8124 6569]
 [ 322  722]]


## Coefficient Importance

In [5]:

clf=model.named_steps['classifier']
feature_names=list(num)+list(cat)
coef=pd.DataFrame({'Feature':feature_names,'Coefficient':clf.coef_[0]})
coef=coef.sort_values('Coefficient',key=abs,ascending=False)
display(coef.head(20))


,Feature,Coefficient
8,X8,-2.628420
1,X1,-1.614711
17,X17,0.871952
3,X3,0.759517
14,X14,0.750271
12,X12,-0.656382
5,X5,0.492418
0,year,-0.381651
15,X15,-0.330117
10,X10,-0.274290


## Save Model

In [6]:

joblib.dump(model,'logistic_regression_baseline.joblib')
print("Saved logistic_regression_baseline.joblib")


Saved logistic_regression_baseline.joblib


## Executive Summary

In [7]:

print("• Logistic Regression provides an interpretable baseline.")
print("• Use these results as the benchmark for all future models.")
print("• Compare Random Forest, XGBoost, LightGBM and CatBoost against this baseline.")


• Logistic Regression provides an interpretable baseline.
• Use these results as the benchmark for all future models.
• Compare Random Forest, XGBoost, LightGBM and CatBoost against this baseline.
